In [1]:
2+2

4

In [2]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
# import torch
import seaborn as sns
import matplotlib.pyplot as plt

import ast
import re
import shutil

In [3]:
# Setting the max characters to be displayed
pd.set_option('display.max_colwidth', None)

# display full dataframe pandas / output window widened
pd.set_option('display.width', 10000) # None

# Setting the precision for float columns
pd.set_option('display.precision', 2)

In [4]:
result = pd.read_parquet("logits_output_meta.parquet")
print(result.shape)
del result['top5_classes']
result.head(4)

(99, 13)


,video_id,cam,session,template,placeholders,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,261008-111314,cam0,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.97,"[69, 81, 128, 124, 78]",True,True
1,261008-111314,cam1,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.89,"[69, 81, 124, 78, 134]",True,True
2,261008-111314,cam2,mime_pretend_poke,Pretending to poke [something],nothing,69,80,Pretending to squeeze [something],0.27,"[80, 69, 74, 128, 122]",False,True
3,261008-111608,cam0,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.60,"[69, 128, 81, 68, 124]",True,True


In [5]:
d = np.load("logits_output.npz", allow_pickle=True)
print(d["logits"].shape)   # (150, 174)
d

(99, 174)


NpzFile 'logits_output.npz' with keys: logits, probs, labels, video_ids, cams...

# main result

In [6]:
from sklearn.metrics import balanced_accuracy_score
    
n = len(result)
top1_acc  = result["correct_top1"].mean()
top5_acc  = result["correct_top5"].mean()
ba_top1   = balanced_accuracy_score(result["label"], result["top1_pred"])

# Top-5 balanced: credit correct if label is anywhere in top-5
top5_pred_adj = [
    row["label"] if row["correct_top5"] else row["top5_preds"][0]
    for _, row in result.iterrows()
]
ba_top5 = balanced_accuracy_score(result["label"], top5_pred_adj)

my_metrics = {
    "total_evaluated":        n,
    "top1_accuracy":          round(top1_acc, 6),
    "top1_balanced_accuracy": round(ba_top1,  6),
    "top5_accuracy":          round(top5_acc, 6),
    "top5_balanced_accuracy": round(ba_top5,  6),
}
my_metrics

/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/user/h.siddiqui/u27673/.conda/envs/vbbf/lib/python3.12/site-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


{'total_evaluated': 99,
 'top1_accuracy': np.float64(0.59596),
 'top1_balanced_accuracy': np.float64(0.603704),
 'top5_accuracy': np.float64(0.888889),
 'top5_balanced_accuracy': np.float64(0.9)}

- cam0 = birds eye
- cam1 = POV
- cam2 = side view

In [7]:
result.groupby("cam")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
)

,top1_acc,top5_acc
cam,,
cam0,0.67,0.94
cam1,0.67,0.97
cam2,0.45,0.76


In [8]:
result.groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Bending [something] so that it deforms,0.56,0.89
Moving [something] and [something] closer to each other,0.72,0.83
Pretending to pick [something] up,0.61,1.00
Pretending to poke [something],0.52,0.78
Pretending to sprinkle air onto [something],0.61,1.00


In [9]:
result[result.cam=='cam1'].groupby("template")[["correct_top1", "correct_top5"]].mean().rename(
        columns={"correct_top1": "top1_acc", "correct_top5": "top5_acc"}
).sort_values("template")

,top1_acc,top5_acc
template,,
Bending [something] so that it deforms,0.83,1.00
Moving [something] and [something] closer to each other,0.83,1.00
Pretending to pick [something] up,0.33,1.00
Pretending to poke [something],0.67,0.89
Pretending to sprinkle air onto [something],0.67,1.00


In [12]:
result[result.cam=='cam1'].groupby(["template","session"]).agg(
    vids = ("correct_top1", "count"),
    top1_acc = ("correct_top1", "mean"),
    top5_acc = ("correct_top5", "mean")
)

vids  top1_acc  top5_acc
template                                                session                                             
Bending [something] so that it deforms                  bending_1                      3      1.00      1.00
                                                        mime_bending                   3      0.67      1.00
Moving [something] and [something] closer to each other mime_moveCloserToEachOther     3      0.67      1.00
                                                        moveCloserToEachOther_1        3      1.00      1.00
Pretending to pick [something] up                       mime_pretend_pickUp            3      0.00      1.00
                                                        pretend_pickUp_1               3      0.67      1.00
Pretending to poke [something]                          mime_pretend_poke              6      0.50      0.83
                                                        pretend_poke_1                 3      1.00      1.00
Pretending to sprinkle air onto [something]             mime_pretend_sprinkle          3      0.33      1.00
                                                        pretend_sprinkle_1             3      1.00      1.00

# Granular analysis

##### id2label

In [13]:
with open('../../data/something-something-v2-id2label.json', 'r') as json_file:
    id2label_dict = json.load(json_file)
    label2id_dict = {v: k for k, v in id2label_dict.items()}


print(list(id2label_dict.items())[0:2])
print(list(label2id_dict.items())[0:2])


[('0', 'Approaching [something] with your camera'), ('1', 'Attaching [something] to [something]')]
[('Approaching [something] with your camera', '0'), ('Attaching [something] to [something]', '1')]


In [14]:
result.head(1)

,video_id,cam,session,template,placeholders,label,top1_pred,top1_class,top1_conf,top5_preds,correct_top1,correct_top5
0,261008-111314,cam0,mime_pretend_poke,Pretending to poke [something],nothing,69,69,Pretending to poke [something],0.97,"[69, 81, 128, 124, 78]",True,True


In [15]:
df = result[['video_id',	'cam'	,'session'	,'placeholders','template',	'label'	,'top1_pred','top1_conf','correct_top1','correct_top5']]
print(df.shape)
df.head(2)

(99, 10)


,video_id,cam,session,placeholders,template,label,top1_pred,top1_conf,correct_top1,correct_top5
0,261008-111314,cam0,mime_pretend_poke,nothing,Pretending to poke [something],69,69,0.97,True,True
1,261008-111314,cam1,mime_pretend_poke,nothing,Pretending to poke [something],69,69,0.89,True,True


In [16]:
# Define class probability column names
class_cols = [f'prob_{i}' for i in range(174)]

# Convert probabilities array into a DataFrame using matching index
probs_df = pd.DataFrame(d['probs'], columns=class_cols, index=df.index)

# Combine metadata with class probabilities
df = pd.concat([df, probs_df], axis=1)

pd.set_option('display.float_format',  '{:,.6f}'.format)

df.head(2)

,video_id,cam,session,placeholders,template,label,top1_pred,top1_conf,correct_top1,correct_top5,...,prob_164,prob_165,prob_166,prob_167,prob_168,prob_169,prob_170,prob_171,prob_172,prob_173
0,261008-111314,cam0,mime_pretend_poke,nothing,Pretending to poke [something],69,69,0.973855,True,True,...,0.000001,0.000003,0.000084,0.000007,0.000023,0.000000,0.000002,0.000000,0.000000,0.000006
1,261008-111314,cam1,mime_pretend_poke,nothing,Pretending to poke [something],69,69,0.889504,True,True,...,0.000003,0.000008,0.000532,0.000062,0.000068,0.000002,0.000010,0.000000,0.000002,0.000029


#### Main Analysis 
Comparison of:
- P(label=3|session=plug_1) -- should be high.
- P(label=3|session=plug_2) -- if its high it means plug as distractor is increasing probabilty of label=3.
- P(label=3|session not like 'plug%')

In [17]:
distractor = 'plug'
label = 3

prob_col = f'prob_{label}'

df['session_temp'] = np.where(df['session'] == f'{distractor}_1', "Normal",
                     np.where(df['session'] == f'{distractor}_2', f"distractor_{distractor}", 'Baseline'))
              
df.groupby(['session','session_temp']).agg(
    videos=('video_id', 'count'),
).reset_index()


,session,session_temp,videos
0,bending_1,Baseline,9
1,mime_bending,Baseline,9
2,mime_moveCloserToEachOther,Baseline,9
3,mime_pretend_pickUp,Baseline,9
4,mime_pretend_poke,Baseline,18
5,mime_pretend_sprinkle,Baseline,9
6,moveCloserToEachOther_1,Baseline,9
7,pretend_pickUp_1,Baseline,9
8,pretend_poke_1,Baseline,9
9,pretend_sprinkle_1,Baseline,9


In [18]:
pd.set_option('display.float_format',  '{:,.5f}'.format)

# Analysis


In [ ]:
def avg_prob_in_diff_cases(df: pd.DataFrame, session_type: str, label: str, label2id_dict: dict, id2label_dict:dict, cams: list):
    label_id = label2id_dict[label]
    prob_col = f'prob_{label_id}'

    df_temp = df[(df['cam'].isin(cams)) & (df['session'].str.contains(session_type))].copy()
    
    r = df_temp.groupby('session').agg(
        videos=('video_id', 'count'),
        prob_mean=(prob_col, 'mean'),
        prob_std=(prob_col, 'std'),
        top1acc=('correct_top1', 'mean'),
        top5acc=('correct_top5', 'mean'),
        examples=('video_id', lambda x: ', '.join(map(str, x.unique()[:3]))),
    ).sort_values('session').reset_index()

    print("=" * 25, session_type, "=" * 25)
    print(f"label {label_id}: {label}")
    print(f"cams used: {cams}")
    print(r)
    print("=" * 20, 'Videos', "=" * 20)

    dist_df = df_temp

    # ------------- chunk to copy videos ------------- #
    source_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/data/hamza_vgd_data/capture/"
    dest_dir = "/user/h.siddiqui/u27673/.project/dir.project/hamza/repos/vgd/zz_temp_dump/"

    # 2. Clear destination directory if it exists, then recreate it
    if os.path.exists(dest_dir):
        shutil.rmtree(dest_dir)
    os.makedirs(dest_dir, exist_ok=True)

    # 3. Get unique video IDs and copy matching files
    unique_video_ids = dist_df['video_id'].dropna().astype(str).unique()

    for filename in os.listdir(source_dir):
        if any(video_id in filename for video_id in unique_video_ids):
            shutil.copy(os.path.join(source_dir, filename), os.path.join(dest_dir, filename))
    # ------------------------------------------------ #

    prob_cols = [c for c in dist_df.columns if c.startswith('prob_')]
    meta_cols = [c for c in dist_df.columns if c not in prob_cols]

    for idx, row in dist_df.iterrows():
        top5_s = row[prob_cols].astype(float).nlargest(5)
        top5_cols = top5_s.index.tolist()
        
        # 1. Print row table as before (meta columns + top 5 prob_X headers)
        print(row[meta_cols + top5_cols].to_frame().T)
        
        # 2. Print mapping and probability cleanly in 5 separate lines
        print("\nTop 5 Probability Breakdown:")
        for col, val in top5_s.items():
            class_id = int(col.split('_')[1])
            class_name = id2label_dict.get(class_id, id2label_dict.get(str(class_id), 'Unknown'))
            print(f"  {col:<8} ->  {val:.5f}  |  {class_name}")

        print(f"\n  {prob_col:<8} ->  {row[prob_col]:.5f}  |  {label}")
            
        print("-" * 50)

In [20]:
avg_prob_in_diff_cases(df, session_type='bending', label='Bending [something] so that it deforms', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= bending =========================
label 2: Bending [something] so that it deforms
cams used: ['cam1']
        session  videos  prob_mean  prob_std  top1acc  top5acc                                     examples
0     bending_1       3    0.74914   0.11323  1.00000  1.00000  261008-114415, 261008-114428, 261008-114453
1  mime_bending       3    0.28158   0.24673  0.66667  1.00000  261008-114331, 261008-114344, 261008-114350
==================== Videos ====================
         video_id   cam       session placeholders                                template label top1_pred top1_conf correct_top1 correct_top5 session_temp  prob_2 prob_143  prob_3 prob_80 prob_65
64  261008-114331  cam1  mime_bending      nothing  Bending [something] so that it deforms     2         2   0.34111         True         True     Baseline 0.34111  0.24956 0.11074 0.10733 0.05397

Top 5 Probability Breakdown:
  prob_2   ->  0.34111  |  Bending [something] so that it deforms
  prob_14

In [21]:
avg_prob_in_diff_cases(df, session_type='Closer', label='Moving [something] and [something] closer to each other', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= Closer =========================
label 37: Moving [something] and [something] closer to each other
cams used: ['cam1']
                      session  videos  prob_mean  prob_std  top1acc  top5acc                                     examples
0  mime_moveCloserToEachOther       3    0.69063   0.50795  0.66667  1.00000  261008-114837, 261008-114847, 261008-114901
1     moveCloserToEachOther_1       3    0.98288   0.01033  1.00000  1.00000  261008-114921, 261008-114942, 261008-114959
==================== Videos ====================
         video_id   cam                     session      placeholders                                                 template label top1_pred top1_conf correct_top1 correct_top5 session_temp prob_37 prob_42 prob_94 prob_38 prob_39
82  261008-114837  cam1  mime_moveCloserToEachOther  nothing, nothing  Moving [something] and [something] closer to each other    37        37   0.98371         True         True     Baseline 0.98371 0.00428 

In [22]:
avg_prob_in_diff_cases(df, session_type='pick', label='Pretending to pick [something] up', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= pick =========================
label 68: Pretending to pick [something] up
cams used: ['cam1']
               session  videos  prob_mean  prob_std  top1acc  top5acc                                     examples
0  mime_pretend_pickUp       3    0.25919   0.09656  0.00000  1.00000  261008-113356, 261008-113406, 261008-113415
1     pretend_pickUp_1       3    0.62820   0.16003  0.66667  1.00000  261008-113428, 261008-113443, 261008-113455
==================== Videos ====================
         video_id   cam              session placeholders                           template label top1_pred top1_conf correct_top1 correct_top5 session_temp prob_81 prob_68 prob_74 prob_80 prob_44
46  261008-113356  cam1  mime_pretend_pickUp      nothing  Pretending to pick [something] up    68        81   0.44691        False         True     Baseline 0.44691 0.37050 0.15445 0.00637 0.00599

Top 5 Probability Breakdown:
  prob_81  ->  0.44691  |  Pretending to take [something] f

In [23]:
avg_prob_in_diff_cases(df, session_type='poke', label='Pretending to poke [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= poke =========================
label 69: Pretending to poke [something]
cams used: ['cam1']
             session  videos  prob_mean  prob_std  top1acc  top5acc                                     examples
0  mime_pretend_poke       6    0.36127   0.39351  0.50000  0.83333  261008-111314, 261008-111608, 261008-111617
1     pretend_poke_1       3    0.89868   0.05273  1.00000  1.00000  261008-112131, 261008-112156, 261008-112220
==================== Videos ====================
        video_id   cam            session placeholders                        template label top1_pred top1_conf correct_top1 correct_top5 session_temp prob_69 prob_81 prob_124 prob_78 prob_56
1  261008-111314  cam1  mime_pretend_poke      nothing  Pretending to poke [something]    69        69   0.88950         True         True     Baseline 0.88950 0.01965  0.01394 0.00845 0.00701

Top 5 Probability Breakdown:
  prob_69  ->  0.88950  |  Pretending to poke [something]
  prob_81  ->  0.019

In [25]:
avg_prob_in_diff_cases(df, session_type='sprinkle', label='Pretending to sprinkle air onto [something]', label2id_dict=label2id_dict, id2label_dict=id2label_dict, cams=['cam1'])

========================= sprinkle =========================
label 79: Pretending to sprinkle air onto [something]
cams used: ['cam1']
                 session  videos  prob_mean  prob_std  top1acc  top5acc                                     examples
0  mime_pretend_sprinkle       3    0.26860   0.24388  0.33333  1.00000  261008-112813, 261008-112823, 261008-112834
1     pretend_sprinkle_1       3    0.90782   0.01854  1.00000  1.00000  261008-112944, 261008-113003, 261008-113021
==================== Videos ====================
         video_id   cam                session placeholders                                     template label top1_pred top1_conf correct_top1 correct_top5 session_temp prob_81 prob_79 prob_68 prob_80 prob_78
28  261008-112813  cam1  mime_pretend_sprinkle      nothing  Pretending to sprinkle air onto [something]    79        81   0.48884        False         True     Baseline 0.48884 0.19751 0.08764 0.06215 0.02936

Top 5 Probability Breakdown:
  prob_81  ->  